In [1]:
from pyspark.sql import SparkSession
Spark=SparkSession.builder\
.appName("Working with files")\
.getOrCreate()

In [2]:
from google.colab import files
uploaded=files.upload()


Saving customers.json to customers.json


In [12]:
df=Spark.read.json(
    "customers.json",
    multiLine=True
)
df.show()
df.printSchema()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|     NULL|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: str

In [13]:
from pyspark.sql.functions import col

flat_customers_df = df.select(
"customer_id",
"name",
"city",
"membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [15]:
#1
df=Spark.read.option(
"multiline",
"true"
).json("customers.json")

In [16]:
#2
df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [17]:
#3
df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [18]:
#4
df.select(
"contact.phone",
"contact.email"
).show()


+----------+---------------+
|     phone|          email|
+----------+---------------+
|9876500011| aarav@mail.com|
|      NULL|  sana@mail.com|
|9876500013|           NULL|
|9876500014|ayesha@mail.com|
|      NULL|           NULL|
+----------+---------------+



In [19]:
#5
df.select(
"preferences.preferred_service",
"preferences.budget_range"
).show()

+-----------------+------------+
|preferred_service|budget_range|
+-----------------+------------+
|           Flight|      Medium|
|            Hotel|        NULL|
|           Flight|        High|
|             NULL|         Low|
|           Flight|        High|
+-----------------+------------+



In [20]:
#6
df.select(
"name",
"city",
"contact.phone",
"contact.email"
).show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [21]:
#7
df.filter(
df.city.isNull()
).show()

+----+------------------+-----------+----------+-----------+--------------+
|city|           contact|customer_id|membership|       name|   preferences|
+----+------------------+-----------+----------+-----------+--------------+
|NULL|{NULL, 9876500013}|          3|      Gold|John Mathew|{High, Flight}|
+----+------------------+-----------+----------+-----------+--------------+



In [22]:
#8
df.filter(
df.contact.phone.isNull()
).show()

+---------+--------------------+-----------+----------+----------+--------------+
|     city|             contact|customer_id|membership|      name|   preferences|
+---------+--------------------+-----------+----------+----------+--------------+
|Bangalore|{sana@mail.com, N...|          2|    Silver| Sana Khan| {NULL, Hotel}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|Vikram Rao|{High, Flight}|
+---------+--------------------+-----------+----------+----------+--------------+



In [23]:
#9
df.filter(
df.contact.email.isNull()
).show()

+------+------------------+-----------+----------+-----------+--------------+
|  city|           contact|customer_id|membership|       name|   preferences|
+------+------------------+-----------+----------+-----------+--------------+
|  NULL|{NULL, 9876500013}|          3|      Gold|John Mathew|{High, Flight}|
|Mumbai|      {NULL, NULL}|          5|  Platinum| Vikram Rao|{High, Flight}|
+------+------------------+-----------+----------+-----------+--------------+



In [24]:
#10
df.filter(
df.membership.isNull()
).show()

+---------+--------------------+-----------+----------+------------+-----------+
|     city|             contact|customer_id|membership|        name|preferences|
+---------+--------------------+-----------+----------+------------+-----------+
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|{Low, NULL}|
+---------+--------------------+-----------+----------+------------+-----------+



In [25]:
#11
df.filter(
df.preferences.preferred_service.isNull()
).show()

+---------+--------------------+-----------+----------+------------+-----------+
|     city|             contact|customer_id|membership|        name|preferences|
+---------+--------------------+-----------+----------+------------+-----------+
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|{Low, NULL}|
+---------+--------------------+-----------+----------+------------+-----------+



In [26]:
#12
df.filter(
df.preferences.budget_range.isNull()
).show()

+---------+--------------------+-----------+----------+---------+-------------+
|     city|             contact|customer_id|membership|     name|  preferences|
+---------+--------------------+-----------+----------+---------+-------------+
|Bangalore|{sana@mail.com, N...|          2|    Silver|Sana Khan|{NULL, Hotel}|
+---------+--------------------+-----------+----------+---------+-------------+



In [27]:
#13
from pyspark.sql.functions import col,sum

df_flat=df.select(
"customer_id",
"name",
"city",
"membership",
col("contact.phone").alias("phone"),
col("contact.email").alias("email"),
col("preferences.preferred_service").alias("preferred_service"),
col("preferences.budget_range").alias("budget_range")
)

df_flat.select([
sum(
col(c).isNull().cast("int")
).alias(c)
for c in df_flat.columns
]).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [28]:
#14
df.fillna({
"city":"Unknown"
}).show()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|  Unknown|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+



In [29]:
#15
df.fillna({
"membership":"Standard"
}).show()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|     NULL|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|  Standard|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+



In [30]:
#16
df_flat.fillna({
"phone":"Not Provided"
}).show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|  9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|  9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|           NULL|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [31]:
#17
df_flat.fillna({
"email":"Not Provided"
}).show()


+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [32]:
#18
df_flat.fillna({
"preferred_service":"Not Selected"
}).show()


+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [33]:
#19
df_flat.fillna({
"budget_range":"Unknown"
}).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [34]:
#20
from pyspark.sql.functions import when

df_status=df_flat.withColumn(
"customer_quality_status",
when(
col("city").isNull()|
col("phone").isNull()|
col("email").isNull()|
col("membership").isNull()|
col("preferred_service").isNull(),
"Incomplete"
).otherwise(
"Complete"
)
)

df_status.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|             Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Fligh

In [35]:
#21
df_status.groupBy(
"customer_quality_status"
).count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+



In [36]:
#22
df_status.filter(
df_status.customer_quality_status==
"Complete"
).show()

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|       name|     city|membership|     phone|         email|preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|          1|Aarav Mehta|Hyderabad|      Gold|9876500011|aarav@mail.com|           Flight|      Medium|               Complete|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+



In [37]:
#23
df_status.filter(
df_status.customer_quality_status==
"Incomplete"
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|             Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|             Incomplete|
+-----------+------------+---------+----------+----------+---------------+----------------

In [38]:
#24
df_flat.fillna({
"membership":"Standard"
}).groupBy(
"membership"
).count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [39]:
#25
df_flat.fillna({
"preferred_service":"Not Selected"
}).groupBy(
"preferred_service"
).count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [40]:
#26
df_flat.write.mode(
"overwrite"
).parquet(
"customers_flat.parquet"
)

In [41]:
#27
df_flat.dropna().write.mode(
"overwrite"
).csv(
"clean_customers.csv",
header=True
)


In [42]:
#28
print(
"Original Count:",
df.count()
)

print(
"Clean Count:",
df_flat.dropna().count()
)

Original Count: 5
Clean Count: 1


In [43]:
#29
df_flat.filter(
col("phone").isNull()|
col("email").isNull()
).show()

+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|       name|     city|membership|     phone|        email|preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|          2|  Sana Khan|Bangalore|    Silver|      NULL|sana@mail.com|            Hotel|        NULL|
|          3|John Mathew|     NULL|      Gold|9876500013|         NULL|           Flight|        High|
|          5| Vikram Rao|   Mumbai|  Platinum|      NULL|         NULL|           Flight|        High|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [44]:
#30
df_flat.filter(
col("preferred_service").isNull()|
col("budget_range").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

